# Market fluctuation analysis

## tl;dr

The executed results below assess whether the shared customer simulator and production pricing engine create visible, volume-responsive, bounded drink-price movement.

## Context & Methods

### Key Assumptions

- Three expected-revenue conditions represent quiet (£2.5k), normal (£10k), and busy (£20k) six-hour services.
- Each service has 48 live drinks, 12 in each category, and 72 five-minute pricing rounds.
- Market crashes are disabled so the measurements isolate ordinary demand-driven movement.
- The primary sample contains 120 seeded services per condition; a separate 40-service-per-condition holdout tests stability.

In [ ]:
import json
from pathlib import Path
import pandas as pd

analysis_dir = Path.cwd()
primary = json.loads((analysis_dir / 'market-fluctuation-results.json').read_text())
holdout = json.loads((analysis_dir / 'market-fluctuation-holdout.json').read_text())
service = pd.DataFrame(primary['serviceSummary'])
category = pd.DataFrame(primary['categorySummary'])
holdout_service = pd.DataFrame(holdout['serviceSummary'])
service

## Data

The source is the repository's current `buildInstantSimulation` path, which combines the shared customer-demand implementation with production market pricing. The tables are generated by `scripts/analyse-market-fluctuations.ts`.

In [ ]:
normal_categories = category[category['scenario'].eq('normal')][[
    'category', 'averageUnitsPerService', 'movingRoundRatePct',
    'averageRoundMovePct', 'averageDistanceFromBasePct',
    'averageProductNightRangePct', 'p90ProductNightRangePct',
    'productsMovedBy60MinutesPct', 'averageReversalsPerProduct'
]].sort_values('averageUnitsPerService', ascending=False)
normal_categories

## Results

Movement should rise with trading volume, stay away from price limits, and remain present in lower-volume categories without fabricating large swings.

In [ ]:
assert list(service['scenario']) == ['quiet', 'normal', 'busy']
assert service['averageRoundMovePct'].is_monotonic_increasing
assert service['averageDistanceFromBasePct'].is_monotonic_increasing
assert service['averageProductNightRangePct'].is_monotonic_increasing
assert service['nearPriceLimitDecisionRatePct'].max() == 0
assert set(normal_categories['category']) == {'Beer', 'Cocktails', 'Spirits', 'Wine'}

comparison = service.merge(holdout_service, on='scenario', suffixes=('_primary', '_holdout'))
comparison['roundMoveDifferencePctPoints'] = (comparison['averageRoundMovePct_primary'] - comparison['averageRoundMovePct_holdout']).abs()
comparison['nightRangeDifferencePctPoints'] = (comparison['averageProductNightRangePct_primary'] - comparison['averageProductNightRangePct_holdout']).abs()
assert comparison['roundMoveDifferencePctPoints'].max() <= 0.02
assert comparison['nightRangeDifferencePctPoints'].max() <= 0.2
comparison[['scenario', 'averageRoundMovePct_primary', 'averageRoundMovePct_holdout', 'averageProductNightRangePct_primary', 'averageProductNightRangePct_holdout']]

## Takeaways

- The engine is decisively volume-responsive: quiet, normal, and busy nights produce progressively larger round moves and full-night ranges.
- Wine is no longer dormant; under normal trade it behaves similarly to cocktails while spirits remain the calmest category.
- No normal pricing decisions approach the configured floor or ceiling.
- The remaining calibration risk is activity frequency: normal and busy nights change prices in nearly every round and produce many direction reversals, which may look jittery even though individual moves are controlled.
- Results are simulation evidence, not a forecast; real venue POS data is still required to calibrate elasticity and category mix.